In [8]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


# Project directories
DATA_DIR: Path = Path("data")
OUTPUT_DIR: Path = Path("output")

# Create the output directory if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Task 1: File ingestion and regular expressions

def read_log_file(file_path: Path) -> tuple[str, str]:
    """
    Read a log file with automatic encoding fallback.

    Attempts UTF-8 decoding first, then ISO-8859-1
    if UTF-8 decoding fails.

    Args:
        file_path: Path to the log file.

    Returns:
        A tuple containing the file contents and
        the encoding used to decode the file.
    """

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            contents = f.read()
        return contents, "utf-8"

    except UnicodeDecodeError:
        with open(file_path, "r", encoding="iso-8859-1") as f:
            contents = f.read()
        return contents, "iso-8859-1"

item_pattern = r"item=(\w+)"
price_pattern = r"price=([0-9]+\.[0-9]{2})"
currency_pattern = r" (USD|EUR|NOK)"
status_pattern = r"\((SUCCESS|REFUND|CANCELLED)\)"

sample_line = (
    "[2026-10-15 08:30:00] [POS_01] "
    "item=Latte price=4.50 EUR (SUCCESS)"
)


test_prices = [
    "price=4.50",
    "price=999.00",
    "price=12.75",
    "price=4",
    "price=invalid",
]

for text in test_prices:
    match = re.search(price_pattern, text)

    if match:
        print(f"{text} -> {float(match.group(1))}")
    else:
        print(f"{text} -> No match")

match = re.search(price_pattern, sample_line)

if match:
    price = float(match.group(1))
    print("Price:", price)
    print("Data type:", type(price))
else:
    print("No price found")

currency_match = re.search(currency_pattern, sample_line)
status_match = re.search(status_pattern, sample_line)

if currency_match:
    print("Currency:", currency_match.group(1))

if status_match:
    print("Status:", status_match.group(1))

timestamp_pattern = (
    r"\[(\d{4}-\d{2}-\d{2} "
    r"\d{2}:\d{2}:\d{2})\]"
)

register_pattern = r"\[(POS_\d{2})\]"

# Combine the first two fields.
combined_pattern = (
    timestamp_pattern
    + r" "
    + register_pattern
    + r" "
    + item_pattern
    + r" "
    + price_pattern
    + currency_pattern
    + r" "
    + status_pattern
)


match = re.fullmatch(combined_pattern, sample_line)

def parse_log(log_text: str) -> list[dict[str, str | float]]:
    """
    Parse register log text into transaction dictionaries.

    Args:
        log_text: Contents of a register log file.

    Returns:
        A list of valid transaction dictionaries.
        Invalid system messages are ignored.
    """
    transactions: list[dict[str, str | float]] = []

    for line in log_text.splitlines():
        match = re.fullmatch(combined_pattern, line)

        if match:
            transaction = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item": match.group(3),
                "price": float(match.group(4)),
                "currency": match.group(5),
                "status": match.group(6),
            }

            transactions.append(transaction)

    return transactions

if match:
    print("Timestamp:", match.group(1))
    print("Register:", match.group(2))
    print("Item:", match.group(3))
    print("Price:", match.group(4))
    print("Currency:", match.group(5))
    print("Status:", match.group(6))
else:
    print("No valid transaction found.")


test_log = """
[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
--- PRINTER PAPER LOW ---
[2026-10-15 08:35:00] [POS_02] item=Espresso price=3.00 NOK (REFUND)
=== NETWORK SYNC OK ===
"""

result = parse_log(test_log)

print("Valid transactions:", len(result))

for transaction in result:
    print(transaction)

# Check the expected results.
assert len(result) == 2
assert result[0]["item"] == "Latte"
assert result[0]["price"] == 4.50
assert result[1]["status"] == "REFUND"



price=4.50 -> 4.5
price=999.00 -> 999.0
price=12.75 -> 12.75
price=4 -> No match
price=invalid -> No match
Price: 4.5
Data type: <class 'float'>
Currency: EUR
Status: SUCCESS
Timestamp: 2026-10-15 08:30:00
Register: POS_01
Item: Latte
Price: 4.50
Currency: EUR
Status: SUCCESS
Valid transactions: 2
{'timestamp': '2026-10-15 08:30:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 08:35:00', 'register_id': 'POS_02', 'item': 'Espresso', 'price': 3.0, 'currency': 'NOK', 'status': 'REFUND'}


In [9]:
# Task 2

from typing import Any
import json

def find_log_files(directory: Path | str, extension: str = ".log") -> list[Path]:
    """Recursively find all files with the given extension (no glob/rglob/os.walk)."""
    directory = Path(directory)
    found: list[Path] = []

    for item in directory.iterdir():
        if item.is_dir():
            found.extend(find_log_files(item, extension))
        elif item.is_file() and item.suffix == extension:
            found.append(item)

    return sorted(found)

def get_active_registers(node: Any) -> set[str]:
    """Recursively collect all register ids where "active" is true."""
    active: set[str] = set()

    if isinstance(node, dict):
        if node.get("active") is True and "id" in node:
            active.add(str(node["id"]))
        for value in node.values():
            active |= get_active_registers(value)
    elif isinstance(node, list):
        for element in node:
            active |= get_active_registers(element)

    return active

In [10]:

# Task 3: Data cleaning and analysis using NumPy


def remove_price_outliers(prices: np.ndarray) -> np.ndarray:
    """
    Detect and replace extreme price outliers.

    Outliers are identified using the three-standard-
    deviation rule and replaced with the median
    of valid prices.

    Args:
        prices: Array containing transaction prices.

    Returns:
        A one-dimensional NumPy array with outliers
        replaced by the median of valid prices.
    """

    # Convert input prices to a one-dimensional float array.
    prices_array = np.asarray(prices, dtype=float).ravel().copy()

    #Handling empty array case
    if prices_array.size == 0:
        return prices_array
    
    prices_mean = np.mean(prices_array)
    prices_std = np.std(prices_array)
    
    # Calculate the absolute distance from the mean.
    deviations = np.abs(prices_array - prices_mean)

    # Identify values more than three standard deviations away.
    outlier_mask = deviations > 3 * prices_std

    # Select the prices that are NOT outliers.
    valid_prices = prices_array[~outlier_mask]

    # Calculate the median of valid prices.
    median_price = np.median(valid_prices)

    # Replace outliers with the median of valid prices.
    prices_array[outlier_mask] = median_price

    return prices_array


def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray,
    ) -> np.ndarray:
    """
    Calculate the refund ratio using NumPy.

    Args:
        sales_count: Array of sales counts per item.
        refund_count: Array of refund counts per item.

    Returns:
        Array containing the refund ratio for each item.
    """

    epsilon = float(1e-6)  # Small constant to avoid division by zero

    denominator = np.array(sales_count) + epsilon
    refund_ratio = np.array(refund_count) / denominator

    return refund_ratio

compute_refund_ratio(np.array([100, 200, 300]), np.array([5, 10, 15]))


array([0.05, 0.05, 0.05])

In [11]:
# Task 3: Test with numeric transaction prices.
outlier_test_prices = np.array([4.50] * 20 + [999.00])

cleaned_prices = remove_price_outliers(outlier_test_prices)

print("Original last price:", outlier_test_prices[-1])
print("Cleaned last price:", cleaned_prices[-1])

assert cleaned_prices[-1] == 4.50

Original last price: 999.0
Cleaned last price: 4.5


In [12]:
# Task 4:

def build_dataframe(records: list[dict[str, str | float]]) -> pd.DataFrame:
    """Convert transaction dicts to a DataFrame indexed by timestamp (sorted)."""
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index()
    return df


# Hourly revenue per item
def hourly_revenue(df: pd.DataFrame) -> pd.Series:
    """Total price per item in 1-hour windows."""
    return df.groupby("item")["price"].resample("1h").sum()


# Pivot table of revenue by item and status
def status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Total revenue per item across SUCCESS, REFUND and CANCELLED."""
    pivot = df.pivot_table(
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0.0,
    )
    return pivot.reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0.0)


# Refund ratio per item
def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Refund ratio per item, computed with compute_refund_ratio (Task 3)."""
    counts = df.pivot_table(
        values="price",
        index="item",
        columns="status",
        aggfunc="count",
        fill_value=0,
    )
    counts = counts.reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)

    ratios = compute_refund_ratio(
        counts["SUCCESS"].to_numpy(),
        counts["REFUND"].to_numpy(),
    )
    return pd.DataFrame({"item": counts.index, "refund_ratio": ratios})

In [13]:

# Debug: Inspect the configuration file.

config_path = DATA_DIR / "store_config.json"

print("File path:", config_path.resolve())
print("File exists:", config_path.exists())
print("File size:", config_path.stat().st_size, "bytes")

# Display the beginning of the file.
with config_path.open("r", encoding="utf-8") as file:
    content = file.read()

print("File contents:", repr(content[:300]))


File path: C:\Users\bonaa\OneDrive\Documents\GitHub\git-deliverables\data\store_config.json
File exists: True
File size: 699 bytes
File contents: '{\n  "company": "Campus Coffee",\n  "regions": [\n    {\n      "region_name": "Main Campus",\n      "stores": [\n        {\n          "name": "Library Cafe",\n          "registers": [\n            {\n              "id": "POS_01",\n              "active": true\n            },\n            {\n              "id": "P'


In [ ]:
# Task 5


def main() -> None:
    """
    Execute the Campus Coffee sales processing pipeline.

    Discovers sales logs, parses transactions,
    cleans prices, generates reports, and saves
    the resulting files.
    """

    log_files: list[Path] = find_log_files(DATA_DIR)

    print(f"Found {len(log_files)} log files.")

    
    # read configuration metadata.
    config_path: Path = DATA_DIR / "store_config.json"

    with config_path.open("r", encoding="utf-8") as file:
        config = json.load(file)

    # TODO: Call get_active_registers()
    # with the configuration object.
    active_registers = get_active_registers(config)

    print("Active registers:", active_registers)



    #Read and parse all transaction logs.

    all_transactions: list[dict[str, str | float]] = []

    for file_path in log_files:

        # Read the current file.
        log_text, encoding_used = read_log_file(file_path)

        # Parse the file contents.
        transactions = parse_log(log_text)

        all_transactions.extend(transactions)

    print(f"Parsed {len(all_transactions)} transactions.")
    print("Number of log files:", len(log_files))
    print("Number of transactions:", len(all_transactions))

    if all_transactions:
        print("First transaction:", all_transactions[0])

    
    #Build the transaction DataFrame.
    df = build_dataframe(all_transactions)

    print("DataFrame shape:", df.shape)
    print(df.head())

    # Convert the price column to a NumPy array.
    prices = df["price"].to_numpy()

    # Apply the outlier cleaning function.
    cleaned_prices = remove_price_outliers(prices)

    # Update the DataFrame with the cleaned prices.
    df["price"] = cleaned_prices

    print("Outlier cleaning complete.")

    #Correct NOK prices.

    # Identify all transactions using NOK.
    nok_mask = df["currency"] == "NOK"
    df.loc[nok_mask, "price"] *= 10.0
    print("Number of NOK transactions:", nok_mask.sum())
    

    # Step 6: Generate summary reports.

    # Step 7: Save the output files.
    

main()


Found 9 log files.
Active registers: {'POS_03', 'POS_02', 'POS_01'}
Parsed 720 transactions.
Number of log files: 9
Number of transactions: 720
First transaction: {'timestamp': '2026-10-15 07:01:02', 'register_id': 'POS_01', 'item': 'Cappuccino', 'price': 4.04, 'currency': 'NOK', 'status': 'SUCCESS'}
DataFrame shape: (720, 5)
                    register_id        item  price currency   status
timestamp                                                           
2026-10-15 07:00:32      POS_02    Sandwich   7.87      NOK  SUCCESS
2026-10-15 07:00:49      POS_03  Cappuccino   3.98      USD  SUCCESS
2026-10-15 07:01:02      POS_01  Cappuccino   4.04      NOK  SUCCESS
2026-10-15 07:01:05      POS_01   Croissant   3.49      USD  SUCCESS
2026-10-15 07:01:13      POS_02    Sandwich   7.38      NOK  SUCCESS
Outlier cleaning complete.
Number of NOK transactions: 236
